# 2) Statistical models: weather, calendar and SARIMA

## Objectives

At the end of this notebook you should be able to:

- build a **regression** forecast of demand from calendar features and a **temperature forecast**, with a flexible (spline) temperature response
- recognise the **weather leak** — using tomorrow's actual temperature instead of its forecast — and measure how much it flatters
- read **residual diagnostics**: why forecast errors persist, and when correcting for that persistence pays
- fit a **SARIMA** model and say what a pure time-series model can and cannot do compared with a regression on the weather

Lesson 1 ended with every baseline failing in the same place: the middle of the day, when demand depends most on that day's temperature — which none of them used. This lesson brings in the weather. The same rule as in lesson 1 applies: only what is known at **noon on the day before** may enter a day-ahead forecast. For the weather that means a **forecast** of tomorrow's temperature, not tomorrow's temperature.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")                       # statsmodels convergence chatter on short SARIMA fits
logging.getLogger("statsmodels").setLevel(logging.ERROR)

### Data, and what is known at noon

In [2]:
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
error = df.temperature_forecast - df.temperature
print(f"day-ahead temperature forecast: MAE {error.abs().mean():.2f} °C, largest miss {error.abs().max():.1f} °C")

dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
train, test = df.loc[:"2024"], df.loc["2025"]
df[["demand", "temperature", "temperature_forecast", "daytype"]].head(3)

day-ahead temperature forecast: MAE 1.28 °C, largest miss 6.7 °C


                          demand  temperature  temperature_forecast  \
2023-01-01 00:00:00  1891.814698     0.758841              0.758841   
2023-01-01 01:00:00  1870.222676     0.439160             -0.223020   
2023-01-01 02:00:00  1867.903308     0.330122             -0.423129   

                               daytype  
2023-01-01 00:00:00  sunday_or_holiday  
2023-01-01 01:00:00  sunday_or_holiday  
2023-01-01 02:00:00  sunday_or_holiday  

The synthetic temperature forecast misses by about 1.3 °C on average, and its errors persist — a forecast that is too warm today tends to be too warm tomorrow, as real weather forecasts are. Holidays are treated like Sundays (lesson 1's exercise showed why), and every feature here — hour, type of day, month, the temperature forecast — is known at noon on D−1.

### Regression, one step at a time

Ordinary least squares with three sets of features. The **temperature response** is a cubic spline: a smooth curve with six degrees of freedom whose shape is learned from the data — we do not tell the model where heating or cooling starts.

In [3]:
SPLINE = "bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
models = {
    "calendar only": "demand ~ C(hour):C(daytype) + C(month)",
    "+ temperature": f"demand ~ C(hour):C(daytype) + C(month) + {SPLINE}",
    "+ temperature x time of day": f"demand ~ C(hour):C(daytype) + C(month) + {SPLINE}:C(hour // 6)",
}
honest = test.assign(temperature=test.temperature_forecast)       # the model sees the forecast, not the truth
rows, fitted = {}, {}
for name, formula in models.items():
    fitted[name] = smf.ols(formula, train).fit()
    err = fitted[name].predict(honest) - test.demand
    leak = fitted[name].predict(test) - test.demand                 # tomorrow's actual temperature: a leak
    rows[name] = {"parameters": len(fitted[name].params), "MAE": err.abs().mean(), "bias": err.mean(),
                  "MAE with actual temperature (leak)": leak.abs().mean()}
regressions = pd.DataFrame(rows).T
regressions.round(1)

                             parameters    MAE  bias  \
calendar only                      83.0  115.7 -30.1   
+ temperature                      89.0   73.6   1.4   
+ temperature x time of day       107.0   72.7   1.3   

                             MAE with actual temperature (leak)  
calendar only                                             115.7  
+ temperature                                              58.6  
+ temperature x time of day                                56.9  

**Calendar alone** — hour × type of day plus month — reaches 115.7 MW, already better than every baseline of lesson 1, but with a bias of −30 MW. **Adding the temperature forecast** halves the error compared with the best baseline of lesson 1 (142.5 MW): **73.6 MW**, and the bias disappears. What looked like drift in lesson 1 was largely weather: 2025 had a different mix of cold and warm days than the training years, and a model that knows the temperature accounts for it. Letting the temperature response differ by time of day buys only 0.9 MW for 18 more parameters — not worth it here.

The last column repeats lesson 1's warning. Fed tomorrow's **actual** temperature, the same model would claim 58.6 MW — 20 % better than it can really do. Weather leaks are a common pitfall: a study evaluated with observed weather reports accuracy no operator could achieve.

In [4]:
m = fitted["+ temperature"]
temps = np.linspace(-5, 35, 81)
grid = pd.DataFrame({"temperature": temps, "hour": 12, "daytype": "workday", "month": 4})
estimated = m.predict(grid) - m.predict(grid.assign(temperature=18.0))
shape_noon = 0.85 + 0.25 * np.sin(np.pi * 6 / 14) + 0.12 * np.exp(-((12 - 19) / 2.2) ** 2)   # the generator's daily shape at 12:00
true = (38 * np.clip(16 - temps, 0, None) + 70 * np.clip(temps - 22, 0, None)) * shape_noon
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(temps, estimated, label="learned by the spline")
ax.plot(temps, true, ls="--", label="how the data were generated")
ax.set(xlabel="temperature (°C)", ylabel="demand change vs 18 °C (MW)", title="Temperature response, workday 12:00")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The spline was never told where heating or cooling starts, yet it recovers both thresholds and the shape between them — where most of the data lie. At the extremes it drifts: it under-estimates heating at −5 °C and over-estimates cooling at 35 °C, because there are few such hours to learn from. That matters more than the average error suggests: the coldest and hottest days are the ones that set the peak, and they are where a statistical model is least reliable.

### Residual diagnostics: errors that persist

In [5]:
resid = (df.demand - m.predict(df)).loc[:"2024"]        # in-sample residuals of the temperature model
acf = pd.Series([resid.autocorr(k) for k in range(1, 73)], index=range(1, 73))
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(acf.index, acf.to_numpy(), width=0.8)
ax.axhline(0, color="black", lw=0.6)
ax.set(xlabel="lag (hours)", ylabel="autocorrelation", title="Residuals of the temperature model")
plt.tight_layout()
plt.show()
print(f"residual autocorrelation: lag 1 {acf[1]:.3f}, lag 12 {acf[12]:.3f}, lag 24 {acf[24]:.3f}")

residual autocorrelation: lag 1 0.905, lag 12 0.258, lag 24 0.074


A good forecast leaves residuals that look like noise. These do not: an error now is almost the same error an hour later (autocorrelation 0.9), fading over about a day. That is useful information — if demand is running 100 MW above the model at 11:00, it will probably still be above it at noon. The simplest way to use it adds the latest known error, shrunk by the lag-1 autocorrelation φ for every hour of lead time: **forecast = model + φᵏ · (last known error)**, k hours ahead. (This is a regression with AR(1) errors — the idea behind ARIMAX.) How much does it help?

In [6]:
phi = acf[1]
everyone = df.assign(temperature=df.temperature_forecast)
base = m.predict(everyone).loc[test.index]
known_error = df.demand - m.predict(df)                    # observed error, known once the hour has passed
by_lead = {}
for k in [1, 2, 3, 6, 12, 24, 36]:
    corrected = base + phi ** k * known_error.shift(k).loc[test.index]
    by_lead[k] = {"model only": (base - test.demand).abs().mean(), "with error correction": (corrected - test.demand).abs().mean()}
by_lead = pd.DataFrame(by_lead).T.rename_axis("hours ahead")
by_lead["gain_%"] = 100 * (1 - by_lead["with error correction"] / by_lead["model only"])
by_lead.round(1)

             model only  with error correction  gain_%
hours ahead                                           
1                  73.6                   50.7    31.0
2                  73.6                   56.0    23.9
3                  73.6                   59.9    18.6
6                  73.6                   66.8     9.2
12                 73.6                   72.1     2.0
24                 73.6                   73.1     0.7
36                 73.6                   73.5     0.1

The correction is a **short-range** tool. One hour ahead it cuts the error by 31 %, six hours ahead by 9 %; at a day-ahead lead time of 13 to 36 hours it is worth almost nothing, because φᵏ has decayed — 0.905¹³ ≈ 0.27, 0.905³⁶ ≈ 0.03. That is why intraday trading and real-time operation lean heavily on the latest measurements, while day-ahead forecasts live or die by their weather input.

### SARIMA: a pure time-series model

A **SARIMA** model forecasts a series from its own past: AR terms for the persistence just seen, a seasonal difference at lag 24 (today minus yesterday at the same hour) and a seasonal AR term. It needs no weather and no calendar. Fitting it on hourly data is slow, so — like a real operator — we refit it on the last six weeks before each forecast, and evaluate on 26 days of 2025 (every second week) rather than all 365. The regression is scored on exactly the same days.

In [7]:
days = pd.date_range("2025-01-08", "2025-12-24", freq="14D")
rows = []
for day in days:
    issue = day - pd.Timedelta(hours=13)                      # 11:00 on D-1: the last known hour
    history = df.demand.loc[issue - pd.Timedelta(days=42):issue]
    sarima = sm.tsa.SARIMAX(history, order=(2, 0, 0), seasonal_order=(1, 1, 0, 24)).fit(disp=False)
    forecast = sarima.forecast(36).to_numpy()[12:]            # hours 00:00-23:00 of day D
    actual = df.demand.loc[day:day + pd.Timedelta(hours=23)]
    rows.append({"day": day, "SARIMA": np.abs(forecast - actual.to_numpy()).mean(),
                 "regression": (base.loc[actual.index] - actual).abs().mean()})
sampled = pd.DataFrame(rows).set_index("day")
print(f"{len(days)} days: SARIMA MAE {sampled.SARIMA.mean():.1f} MW, temperature regression {sampled.regression.mean():.1f} MW; "
      f"SARIMA better on {(sampled.SARIMA < sampled.regression).sum()} of {len(days)} days")

26 days: SARIMA MAE 96.2 MW, temperature regression 76.0 MW; SARIMA better on 10 of 26 days


SARIMA, using nothing but past demand, reaches 96.2 MW on the 26 sampled days — far better than lesson 1's baselines (about 140 MW), because it learns the recent level and the daily shape. It is still clearly behind the temperature regression on the same days (76.0 MW): it can only extrapolate the past, so it cannot see a cold front coming. It wins on 10 of the 26 days — worth examining: plausible reasons are days on which the temperature forecast missed badly, or on which the weather barely changed, so that recent demand was the best guide.

That sums up the trade-off. A SARIMA model needs only the series itself and adapts to recent conditions; a regression needs good explanatory data — here a temperature forecast — and rewards it. In practice the two ideas are combined: a regression on weather and calendar, with time-series terms for what the regressors miss. Lesson 3 does the same with machine learning.

## Check your understanding

**Regression**

1. The calendar-only model has a bias of about −30 MW in 2025, the temperature model almost none. Why does adding the weather remove a bias that looked like drift?
2. Why can a spline learn the heating and cooling thresholds by itself, and what would go wrong with a single straight line in temperature?

**Time-series models**

1. The error correction cuts the 1-hour-ahead error by almost a third but does nothing for a day-ahead forecast. Explain with φᵏ.
2. When would you prefer a SARIMA model to a weather regression? Think of data you might not have.